### Silver: `02_silver.taxi_zone_lookup` — typed lookup + `is_physical` flag
Non-physical rows (Borough Unknown/N/A, Zone N/A / Outside of NYC → IDs 264, 265) are kept but flagged,
so trips there stay valid while the stress index only covers physical zones.

In [0]:
%run ../../config/config

In [0]:
from pyspark.sql.functions import col, trim

df = spark.table(tbl("01_bronze", "taxi_zone_lookup_raw")).select(
    col("LocationID").cast("int").alias("location_id"),
    trim(col("Borough")).alias("borough"),
    trim(col("Zone")).alias("zone"),
    trim(col("service_zone")).alias("service_zone"),
)
df = df.withColumn("is_physical", ~col("borough").isin(NON_PHYSICAL_BOROUGHS) & ~col("zone").isin(NON_PHYSICAL_ZONES))

assert df.count() == df.select("location_id").distinct().count(), "duplicate location_id in lookup"
df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(tbl("02_silver", "taxi_zone_lookup"))

In [0]:
display(spark.sql(f"SELECT * FROM {tbl('02_silver', 'taxi_zone_lookup')} WHERE NOT is_physical"))